# Lab 04 — AI_CLASSIFY & AI_SENTIMENT

Two complementary functions for understanding text: **classify** assigns a category from a provided list; **sentiment** returns a score from -1 (negative) to +1 (positive).

| Function | Returns | Use Case |
|---|---|---|
| `AI_CLASSIFY` | Label + confidence score | Categorize orders, tickets, feedback |
| `AI_SENTIMENT` | Score -1 to +1 | Measure customer satisfaction at scale |


## Step 1 — Environment Setup

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — AI_CLASSIFY: Categorize Order Comments

Classify free-text comments into business categories — no training needed.

In [ ]:
CREATE OR REPLACE VIEW orders_sample AS
SELECT
    o_orderkey AS order_key, o_orderstatus AS order_status,
    o_totalprice AS total_price, o_orderdate AS order_date,
    o_orderpriority AS order_priority, o_comment AS order_comment
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
LIMIT 500;

In [ ]:
SELECT
    order_key,
    order_comment,
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        order_comment,
        ['shipping_issue', 'pricing_concern', 'general_inquiry', 'urgent_request', 'positive_feedback']
    ) AS classification
FROM orders_sample
LIMIT 20;

In [ ]:
WITH classified AS (
    SELECT
        order_priority,
        SNOWFLAKE.CORTEX.AI_CLASSIFY(
            order_comment,
            ['shipping_issue', 'pricing_concern', 'general_inquiry', 'urgent_request', 'positive_feedback']
        ):label::STRING AS category
    FROM orders_sample
)
SELECT category, COUNT(*) AS cnt
FROM classified
GROUP BY category
ORDER BY cnt DESC;

## Step 3 — AI_SENTIMENT: Product Review Scoring

Returns a continuous score from -1 (negative) to +1 (positive). Works on any text with no training.

In [ ]:
CREATE OR REPLACE TABLE product_reviews (
    review_id INT AUTOINCREMENT, product_name VARCHAR(200),
    category VARCHAR(100), review_text TEXT, star_rating FLOAT
);

INSERT INTO product_reviews (product_name, category, review_text, star_rating) VALUES
('Wireless Headphones Pro', 'Electronics', 'Absolutely love these headphones! Crystal clear sound and the noise cancellation is incredible.', 5.0),
('Wireless Headphones Pro', 'Electronics', 'Sound quality is decent but they hurt my ears after an hour. Padding needs improvement.', 3.0),
('Wireless Headphones Pro', 'Electronics', 'Terrible battery life. Dies after 2 hours despite claiming 20 hours. False advertising.', 1.0),
('Running Shoes Ultra', 'Sports', 'These shoes transformed my running experience. Super lightweight and great arch support.', 5.0),
('Running Shoes Ultra', 'Sports', 'Good shoes but the sizing runs small. Had to exchange for a larger pair.', 3.0),
('Running Shoes Ultra', 'Sports', 'Sole came apart after just two weeks of use. Very disappointing quality.', 1.0),
('Smart Home Hub', 'Electronics', 'Setup was a breeze and it works perfectly with all my devices. Voice control is very responsive.', 5.0),
('Smart Home Hub', 'Electronics', 'Works okay but the app is clunky and crashes frequently on Android.', 2.0),
('Smart Home Hub', 'Electronics', 'Constantly loses connection to WiFi. Had to restart it multiple times daily.', 1.0);

In [ ]:
SELECT
    product_name, star_rating,
    ROUND(SNOWFLAKE.CORTEX.AI_SENTIMENT(review_text), 3) AS sentiment_score,
    LEFT(review_text, 80) AS review_snippet
FROM product_reviews
ORDER BY sentiment_score DESC;

In [ ]:
SELECT
    product_name,
    COUNT(*) AS review_count,
    ROUND(AVG(star_rating), 2) AS avg_stars,
    ROUND(AVG(SNOWFLAKE.CORTEX.AI_SENTIMENT(review_text)), 3) AS avg_sentiment
FROM product_reviews
GROUP BY product_name
ORDER BY avg_sentiment DESC;

## Step 4 — Find Star-Sentiment Mismatches

Detect reviews where the star rating doesn't match the text sentiment — potential gaming or errors.

In [ ]:
SELECT
    product_name, star_rating,
    ROUND(SNOWFLAKE.CORTEX.AI_SENTIMENT(review_text), 3) AS sentiment_score,
    review_text
FROM product_reviews
WHERE ABS(star_rating / 5.0 - (SNOWFLAKE.CORTEX.AI_SENTIMENT(review_text) + 1) / 2) > 0.3
ORDER BY star_rating DESC;

## Key Takeaways

- `AI_CLASSIFY` returns JSON with `label` and `score` — any business categories, no training.
- `AI_SENTIMENT` returns a continuous score (-1 to +1) — great for ranking and thresholds.
- Combine with `GROUP BY` for instant analytics dashboards.
- Compare AI sentiment vs. star ratings to detect gaming or mismatched feedback.
